1️⃣ Architecture Overview
User
  ↓
OpenAI Model (Agent)
  ↓
MCP Client (Python)
  ↓
MCP Server (Tool Provider)
  ↓
MySQL Database

CREATE DATABASE company;

USE company;

CREATE TABLE employees (
    id INT PRIMARY KEY AUTO_INCREMENT,
    name VARCHAR(100),
    department VARCHAR(100),
    salary INT
);

INSERT INTO employees (name, department, salary) VALUES
('Amit Sharma', 'Engineering', 120000),
('Priya Verma', 'Engineering', 95000),
('Rahul Gupta', 'Marketing', 80000),
('Sneha Patil', 'HR', 70000),
('Vikram Singh', 'Engineering', 150000),
('Neha Kulkarni', 'Finance', 110000),
('Arjun Mehta', 'Sales', 90000),
('Pooja Nair', 'Marketing', 85000),
('Rohan Deshmukh', 'Engineering', 105000),
('Kavita Joshi', 'HR', 72000);

Python MCP Tool for MySQL

In [1]:
pip install openai mysql-connector-python fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import os

print("Python version:", sys.version)
print("Python executable:", sys.executable)

try:
    from mcp.server.fastmcp import FastMCP
    print("✅ FastMCP is now available!")
except ImportError as e:
    print("❌ FastMCP module not found.")
    print("Error:", e)
    print("\nTry these steps:")
    print("1. Install MCP: pip install mcp")
    print("2. Restart the kernel or Python environment")
    print("3. Verify installation using: pip show mcp")

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Python executable: D:\Ethans\Python\AI-Automation\myenv\Scripts\python.exe
✅ FastMCP is now available!


import mysql.connector
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("mysql-server")

def run_query(query: str):
    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="",
        database="company"
    )

    cursor = conn.cursor(dictionary=True)
    cursor.execute(query)

    result = cursor.fetchall()

    cursor.close()
    conn.close()

    return result


@mcp.tool()
def query_mysql(query: str):
    """Execute a SQL query on MySQL database"""
    return run_query(query)


if __name__ == "__main__":
    mcp.run()

Python Client Using OpenAI + MCP

In [3]:
import asyncio
import os
import json
from openai import AsyncOpenAI
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

client = AsyncOpenAI() 

# Use the absolute paths we created
employee_path = os.path.abspath("employeeserver.py").replace("\\", "/")
python_path = os.path.abspath("myenv/Scripts/python.exe").replace("\\", "/")

server = StdioServerParameters(
    command=python_path,
    args=[employee_path], # Removed the stray ')'
    cwd=os.getcwd()
)

async def main():
    async with stdio_client(server) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            mcp_tools = await session.list_tools()
            
            openai_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.inputSchema,
                    },
                }
                for tool in mcp_tools.tools
            ]

            # First Call: Ask the LLM
            messages = [{"role": "user", "content": "Show all employees in engineering department"}]
            response = await client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                tools=openai_tools
            )

            message = response.choices[0].message
            
            # Second Step: If the LLM wants to use a tool, execute it via MCP
            if message.tool_calls:
                for tool_call in message.tool_calls:
                    tool_name = tool_call.function.name
                    tool_args = json.loads(tool_call.function.arguments)
                    
                    # Execute tool call on the MCP server
                    result = await session.call_tool(tool_name, tool_args)
                    
                    # Print the raw data or feed it back to the LLM
                    print(f"Tool Output: {result.content}")
            else:
                print(message.content)

# Run in Notebook
await main()

UnsupportedOperation: fileno